In [1]:
# ==============================================================================
# GERADOR SINTÉTICO DE DADOS - SAC MÓVEIS RESIDENCIAIS
# ==============================================================================
import pandas as pd
import random

templates = {
    'vendas': {
        's': ['', 'Olá', 'Bom dia', 'Gostaria de saber', 'Por favor'],
        'a': ['quero comprar', 'qual o preco do', 'tem cupom para', 'como faco para adquirir', 'desejo orcamento de'],
        'o': ['sofa retratil 3 lugares', 'conjunto de mesa de jantar', 'guarda roupa casal', 'painel para tv', 'colchao queen size']
    },
    'suporte': {
        's': ['', 'Oi', 'Preciso de ajuda', 'Por gentileza', 'Socorro'],
        'a': ['como montar o', 'onde baixo o manual do', 'estou com duvida no', 'veio faltando parafuso no', 'preciso de assistencia para'],
        'o': ['armario de cozinha', 'rack da sala', 'berco do bebe', 'esquema de montagem', 'manual da estante']
    },
    'trocas_devolucoes': {
        's': ['', 'Olá', 'Por favor', 'Gostaria de solicitar', 'Quero abrir'],
        'a': ['preciso trocar o', 'quero devolver a', 'como solicito o estorno do', 'desejo solicitar a troca da', 'como funciona a devolucao do'],
        'o': ['produto com defeito', 'mesa que veio arranhada', 'cadeira no prazo de 7 dias', 'pedido cancelado', 'item com avaria']
    },
    'reclamacoes': {
        's': ['', 'Urgente', 'Pessimo atendimento', 'Absurdo', 'Quero registrar'],
        'a': ['estou indignado com o', 'quero fazer uma queixa do', 'estou reclamando do', 'produto veio quebrado e o', 'atendimento horrivel do'],
        'o': ['atraso na minha entrega', 'servico de montagem', 'sac que nao responde', 'pos venda da loja', 'estado do meu movel']
    },
    'logistica_entregas': {
        's': ['', 'Olá', 'Bom dia', 'Por gentileza', 'Preciso saber'],
        'a': ['onde esta o meu', 'qual o prazo de entrega do', 'como rastreio a', 'qual a transportadora do', 'quando chega o'],
        'o': ['meu pedido', 'codigo de rastreamento', 'movel comprado', 'status do envio', 'agendamento da entrega']
    }
}

amostras = []
random.seed(42)

for intencao, comp in templates.items():
    for _ in range(20):  # Total: 100 amostras (20 por classe)
        s = random.choice(comp['s'])
        a = random.choice(comp['a'])
        o = random.choice(comp['o'])
        frase = f"{s} {a} {o}".strip().capitalize()
        amostras.append({'texto': frase, 'intencao': intencao})

df_moveis = pd.DataFrame(amostras)
df_moveis.to_csv('dataset_moveis_100.csv', index=False, encoding='utf-8')

print(" Dataset 'dataset_moveis_100.csv' criado com 100 frases distribuidas em 5 intencoes!")

 Dataset 'dataset_moveis_100.csv' criado com 100 frases distribuidas em 5 intencoes!


In [2]:
# ==============================================================================
# ATIVIDADE 1: CHATBOT VERSÃO 1 (KNN)
# ==============================================================================
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

# 1. Carregar dataset do CSV
df = pd.read_csv('dataset_moveis_100.csv')

# 2. Divisão Treino e Teste
X_train, X_test, y_train, y_test = train_test_split(
    df['texto'], df['intencao'], test_size=0.30, random_state=42, stratify=df['intencao']
)

# TODO 1: Monte a Pipeline utilizando TfidfVectorizer e KNeighborsClassifier(n_neighbors=3, metric='cosine')
pipeline_knn = Pipeline([
    ('vectorizer', TfidfVectorizer()),
    ('classifier', KNeighborsClassifier(n_neighbors=3, metric='cosine'))
])

# TODO 2: Treine a pipeline com os dados de treino (X_train, y_train)
pipeline_knn.fit(X_train, y_train)

# TODO 3: Gere as predicoes nos dados de teste e exiba o classification_report e a confusion_matrix
y_pred = pipeline_knn.predict(X_test)

print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

LIMIAR_CONFIANCA = 0.50

print("\n=== INICIANDO BATERIA DE TESTES (10 INPUTS OBRIGATÓRIOS) ===")

for i in range(1, 11):
    print(f"\n[Teste {i}/10]")

    # TODO 4: Solicite a frase do usuario via teclado
    frase = input("Digite a frase do cliente: ").strip()

    # TODO 5: Extraia as probabilidades e a classe prevista usando predict_proba e predict
    probs = pipeline_knn.predict_proba([frase])[0]
    maior_prob = np.max(probs)
    intencao = pipeline_knn.predict([frase])[0]

    # TODO 6: Aplique a regra de decisao:
    # Se maior_prob >= LIMIAR_CONFIANCA: imprima a intencao e a probabilidade.
    # Senao: imprima o Fallback encaminhando para atendimento humano.

    if maior_prob >= LIMIAR_CONFIANCA:
        print(f"Intenção identificada: {intencao}")
        print(f"Confiança: {maior_prob:.2%}")
    else:
        print("FALLBACK: Não consegui identificar sua solicitação.")
        print("Encaminhando o cliente para a equipe humana.")


                    precision    recall  f1-score   support

logistica_entregas       1.00      1.00      1.00         6
       reclamacoes       1.00      1.00      1.00         6
           suporte       1.00      1.00      1.00         6
 trocas_devolucoes       1.00      1.00      1.00         6
            vendas       1.00      1.00      1.00         6

          accuracy                           1.00        30
         macro avg       1.00      1.00      1.00        30
      weighted avg       1.00      1.00      1.00        30

[[6 0 0 0 0]
 [0 6 0 0 0]
 [0 0 6 0 0]
 [0 0 0 6 0]
 [0 0 0 0 6]]

=== INICIANDO BATERIA DE TESTES (10 INPUTS OBRIGATÓRIOS) ===

[Teste 1/10]
Digite a frase do cliente: Quero saber sobre uma coisa que não tem nada a ver com móveis, podem me ajudar?
Intenção identificada: reclamacoes
Confiança: 66.67%

[Teste 2/10]
Digite a frase do cliente: palmeiras nao tem mundial
Intenção identificada: reclamacoes
Confiança: 66.67%

[Teste 3/10]
Digite a frase do cli

In [3]:
# ==============================================================================
# ATIVIDADE 2: CHATBOT VERSÃO 2 - DECISION TREE
# ==============================================================================

import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split


# ==============================================================================
# 1. CARREGAR O DATASET
# ==============================================================================

df = pd.read_csv('dataset_moveis_100.csv')

print("Dataset carregado com sucesso!")
print(f"Total de registros: {len(df)}")


# ==============================================================================
# 2. DIVISÃO ESTRATIFICADA TREINO/TESTE
# ==============================================================================

X_train, X_test, y_train, y_test = train_test_split(
    df['texto'],
    df['intencao'],
    test_size=0.30,
    random_state=42,
    stratify=df['intencao']
)


# ==============================================================================
# 3. CRIAR PIPELINE: TF-IDF + DECISION TREE
# ==============================================================================

pipeline_tree = Pipeline([
    ('vectorizer', TfidfVectorizer()),
    ('classifier', DecisionTreeClassifier(random_state=42))
])


# ==============================================================================
# 4. TREINAR O MODELO
# ==============================================================================

pipeline_tree.fit(X_train, y_train)

print("\nModelo Decision Tree treinado com sucesso!")


# ==============================================================================
# 5. FAZER PREVISÕES NO CONJUNTO DE TESTE
# ==============================================================================

y_pred = pipeline_tree.predict(X_test)


# ==============================================================================
# 6. MATRIZ DE CONFUSÃO
# ==============================================================================

print("\n" + "=" * 70)
print("MATRIZ DE CONFUSÃO")
print("=" * 70)

print(confusion_matrix(y_test, y_pred))


# ==============================================================================
# 7. RELATÓRIO DE CLASSIFICAÇÃO
# ==============================================================================

print("\n" + "=" * 70)
print("RELATÓRIO DE CLASSIFICAÇÃO")
print("=" * 70)

print(classification_report(y_test, y_pred))


# ==============================================================================
# 8. LIMIAR DE CONFIANÇA
# ==============================================================================

LIMIAR_CONFIANCA = 0.50


# ==============================================================================
# 9. TESTES MANUAIS - 8 INPUTS OBRIGATÓRIOS
# ==============================================================================

print("\n" + "=" * 70)
print("INICIANDO BATERIA DE TESTES - 8 INPUTS")
print("=" * 70)

for i in range(1, 9):

    print(f"\n[Teste {i}/8]")

    # Solicitar frase ao usuário
    frase = input("Digite a frase do cliente: ").strip()

    # Prever a intenção
    intencao = pipeline_tree.predict([frase])[0]

    # Obter as probabilidades
    probs = pipeline_tree.predict_proba([frase])[0]

    # Encontrar a maior probabilidade
    maior_prob = np.max(probs)

    # --------------------------------------------------------------------------
    # 10. REGRA DE FALLBACK
    # --------------------------------------------------------------------------

    if maior_prob >= LIMIAR_CONFIANCA:

        print(f"Intenção identificada: {intencao}")
        print(f"Confiança: {maior_prob:.2%}")

    else:

        print(
            "Desculpe, não entendi sua solicitação. "
            "Encaminhando você para um atendente humano..."
        )


Dataset carregado com sucesso!
Total de registros: 100

Modelo Decision Tree treinado com sucesso!

MATRIZ DE CONFUSÃO
[[4 0 0 0 2]
 [1 4 1 0 0]
 [0 0 6 0 0]
 [0 0 1 5 0]
 [0 0 0 1 5]]

RELATÓRIO DE CLASSIFICAÇÃO
                    precision    recall  f1-score   support

logistica_entregas       0.80      0.67      0.73         6
       reclamacoes       1.00      0.67      0.80         6
           suporte       0.75      1.00      0.86         6
 trocas_devolucoes       0.83      0.83      0.83         6
            vendas       0.71      0.83      0.77         6

          accuracy                           0.80        30
         macro avg       0.82      0.80      0.80        30
      weighted avg       0.82      0.80      0.80        30


INICIANDO BATERIA DE TESTES - 8 INPUTS

[Teste 1/8]
Digite a frase do cliente: Quero saber sobre uma coisa que não tem nada a ver com móveis, podem me ajudar? Intenção: reclamacoes
Intenção identificada: trocas_devolucoes
Confiança: 100.00%

[